# GL-02 Node Classification on the Karate Club Graph

Audience:
- Learners studying a first graph-learning classification lab without PyG or DGL.

Learning goals:
- load a standard graph dataset in a dependency-light form;
- compare a feature-only baseline against a graph-aware model;
- train a small two-layer GCN-style classifier from scratch; and
- inspect train, validation, and test behavior on a node-classification task.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path
from sklearn.linear_model import LogisticRegression


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
MODULE_SRC = REPO_ROOT / "modules" / "13-graph-learning" / "src"
if str(MODULE_SRC) not in sys.path:
    sys.path.insert(0, str(MODULE_SRC))

from graph_learning_utils import accuracy, karate_club_data, normalized_laplacian, train_two_layer_gcn


## Step 1 - Load the Karate Club graph

Zachary's Karate Club graph is a standard small benchmark for relational learning. Here we use one-hot node identities as features so the graph structure does the real work.


In [ ]:
dataset = karate_club_data()
adjacency = dataset["adjacency"]
features = dataset["features"]
labels = dataset["labels"]
train_mask = dataset["train_mask"]
val_mask = dataset["val_mask"]
test_mask = dataset["test_mask"]

{
    "num_nodes": int(adjacency.shape[0]),
    "num_edges": int(adjacency.sum() // 2),
    "class_balance": {"class_0": int((labels == 0).sum()), "class_1": int((labels == 1).sum())},
    "train_nodes": np.where(train_mask)[0].tolist(),
    "val_nodes": np.where(val_mask)[0].tolist(),
}


## Step 2 - Fit a feature-only baseline

Because the features are one-hot node IDs, a logistic baseline trained on only a few labeled nodes has very little basis for generalization. This gives us a clean comparison against a graph-aware model.


In [ ]:
baseline = LogisticRegression(max_iter=2000, random_state=0)
baseline.fit(features[train_mask], labels[train_mask])
baseline_predictions = baseline.predict(features)

{
    "train_accuracy": accuracy(baseline_predictions, labels, train_mask),
    "val_accuracy": accuracy(baseline_predictions, labels, val_mask),
    "test_accuracy": accuracy(baseline_predictions, labels, test_mask),
    "all_nodes_accuracy": float((baseline_predictions == labels).mean()),
}


## Step 3 - Train a small GCN-style model

The implementation uses the propagation rule $\hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}$ and trains a two-layer network with ReLU and masked cross-entropy on the labeled nodes.


In [ ]:
training_run = train_two_layer_gcn(
    adjacency=adjacency,
    features=features,
    labels=labels,
    train_mask=train_mask,
    val_mask=val_mask,
    hidden_dim=16,
    learning_rate=0.6,
    weight_decay=5e-4,
    epochs=400,
    seed=7,
)

gcn_predictions = training_run["predictions"]
training_run["history"]


## Step 4 - Evaluate the graph-aware classifier

The graph model can propagate signal from the labeled nodes through the relational structure, so it usually performs much better than the feature-only baseline on unlabeled nodes.


In [ ]:
comparison = {
    "baseline": {
        "train_accuracy": accuracy(baseline_predictions, labels, train_mask),
        "val_accuracy": accuracy(baseline_predictions, labels, val_mask),
        "test_accuracy": accuracy(baseline_predictions, labels, test_mask),
        "all_nodes_accuracy": float((baseline_predictions == labels).mean()),
    },
    "gcn": {
        "train_accuracy": accuracy(gcn_predictions, labels, train_mask),
        "val_accuracy": accuracy(gcn_predictions, labels, val_mask),
        "test_accuracy": accuracy(gcn_predictions, labels, test_mask),
        "all_nodes_accuracy": float((gcn_predictions == labels).mean()),
    },
}
comparison


## Step 5 - Visualize the learned separation in a spectral embedding

For a simple plot, we use the second and third eigenvectors of the normalized Laplacian as a two-dimensional graph embedding and color points by the GCN predictions.


In [ ]:
laplacian = normalized_laplacian(adjacency)
eigenvalues, eigenvectors = np.linalg.eigh(laplacian)
embedding = eigenvectors[:, 1:3]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True, sharey=True)
for ax, values, title in [
    (axes[0], labels, "true labels"),
    (axes[1], gcn_predictions, "GCN predictions"),
]:
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=values, cmap="coolwarm", s=70, edgecolor="black")
    for index, (x_coord, y_coord) in enumerate(embedding):
        ax.text(x_coord + 0.01, y_coord + 0.01, str(index), fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("eigenvector 2")

axes[0].set_ylabel("eigenvector 3")
fig.colorbar(scatter, ax=axes.ravel().tolist(), shrink=0.85)
fig.tight_layout()
plt.close(fig)

{"smallest_eigenvalues": np.round(eigenvalues[:5], 4).tolist()}


## Interpretation

This notebook isolates the value of relational inductive bias. With identity features alone, the baseline has almost no way to generalize. The GCN-style model succeeds because it composes local neighborhood information through the graph.

Suggested extensions:
- vary the train-mask size and record how accuracy changes;
- replace the GCN propagation operator with row-normalized averaging and compare results; and
- add a third propagation step to observe whether oversmoothing begins to reduce discrimination.
